In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

In [ ]:
REBUILD_DATA = True

class DogsVSCats():
  IMG_SIZE = 224
  CATS = "/content/drive/My Drive/CatsAndDogs/PetImages/Cat"
  DOGS = "/content/drive/My Drive/CatsAndDogs/PetImages/Dog"
  LABELS = {CATS: 0, DOGS: 1}
  training_data = []
  
  catcount = 0
  dogcount = 0
  
  def make_training_data(self):
    for label in self.LABELS:
      for f in tqdm(os.listdir(label)):
        if "jpg" in f:
          try:
            path = os.path.join(label, f)
            img = cv2.imread(path)
            img = cv2.resize(img, (self.IMG_SIZE, self.IMG_SIZE))
            self.training_data.append([np.array(img), np.eye(2)[self.LABELS[label]]])
            
            if label == self.CATS:
              self.catcount += 1
            if label == self.DOGS:
              self.dogcount += 1
              
          except Exception as e:
            pass  
          
    np.random.shuffle(self.training_data)
    np.save("training_data.npy", self.training_data)
    print("Cats:", self.catcount)
    print("dogs:", self.dogcount)
  
if REBUILD_DATA:
  dogsvscats = DogsVSCats()
  dogsvscats.make_training_data()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
import random
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Flatten, Conv2D, MaxPooling2D

In [ ]:
keras.backend.set_image_data_format('channels_last')

In [ ]:
pickle_in = open("/content/drive/My Drive/CatsAndDogs/PetImages/X.pickle","rb")
X = pickle.load(pickle_in)

pickle_in = open("/content/drive/My Drive/CatsAndDogs/PetImages/y.pickle","rb")
y = pickle.load(pickle_in)

In [ ]:
np.array(X).reshape(-1, 100, 100, 1).shape

In [ ]:
np.array(y).shape

In [ ]:
IMG_SIZE = 100

X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array(y)

In [ ]:
X = X/255.0

In [ ]:
try: del(model) 
except Exception: pass

model = keras.Sequential()

model.add(layers.Conv2D(filters = 64, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_1a', input_shape = X.shape[1:]))
model.add(layers.Conv2D(filters = 64, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_1b'))
model.add(layers.MaxPool2D(pool_size = (2, 2), strides = 2, name = 'MaxPool2D_1a'))

model.add(layers.Conv2D(filters = 128, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_2a'))
model.add(layers.Conv2D(filters = 128, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_2b'))
model.add(layers.MaxPool2D(pool_size = (2, 2), strides = 2, name = 'MaxPool2D_2a'))

model.add(layers.Conv2D(filters = 256, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_3a'))
model.add(layers.Conv2D(filters = 256, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_3b'))
model.add(layers.Conv2D(filters = 256, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_3c'))
model.add(layers.MaxPool2D(pool_size = (2, 2), strides = 2, name = 'MaxPool2D_3a'))

model.add(layers.Conv2D(filters = 512, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_4a'))
model.add(layers.Conv2D(filters = 512, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_4b'))
model.add(layers.Conv2D(filters = 512, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_4c'))
model.add(layers.MaxPool2D(pool_size = (2, 2), strides = 2, name = 'MaxPool2D_4a'))

model.add(layers.Conv2D(filters = 512, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_5a'))
model.add(layers.Conv2D(filters = 512, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_5b'))
model.add(layers.Conv2D(filters = 512, kernel_size = (3, 3), strides = 1, padding = 'same', activation="relu", name = 'Conv2d_5c'))
model.add(layers.MaxPool2D(pool_size = (2, 2), strides = 2, name = 'MaxPool2D_5a'))

model.add(layers.Flatten(name = 'flatten'))

model.add(layers.Dense(units = 4096, activation = "relu", name = 'fc1'))
model.add(layers.Dense(units = 4096, activation = "relu", name = 'fc2'))
model.add(layers.Dense(units = 1, activation = 'sigmoid', name = 'fc3'))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam', loss = 'binary_crossentropy', metrics=['accuracy'])

In [ ]:
# from keras.callbacks import ModelCheckpoint, EarlyStopping
# checkpoint = ModelCheckpoint("vgg16_1.h5", monitor='val_acc', verbose=1, save_best_only=True, save_weights_only=False, mode='auto', period=1)
# early = EarlyStopping(monitor='val_acc', min_delta=0, patience=20, verbose=1, mode='auto')
# hist = model.fit_generator(steps_per_epoch=100,generator=traindata, validation_data= testdata, validation_steps=10,epochs=100,callbacks=[checkpoint,early])

In [ ]:
model.fit(X, y, batch_size=32, epochs = 3)

In [ ]:
from keras.applications.vgg16 import VGG16
vggmodel = VGG16(weights='imagenet', include_top=True)

In [ ]:
vggmodel.summary()

In [ ]:
for l in (vggmodel.layers)[:19]:
    #print(l)
    l.trainable = False

In [ ]:
vggmodel.input.set_shape(X.shape[1:])

In [ ]:
vggmodel.input.set_shape([100, 100, 1])

In [ ]:
from keras.models import Model
fc2 = vggmodel.layers[-2].output
fc3 = layers.Dense(1, activation = "sigmoid")(fc2)
model_final = Model(inputs = vgg_modified_input, outputs = fc3)